In [1]:
import pandas as pd
import datetime as dt
from datetime import datetime
from dataretrieval import nwis
from dataretrieval import waterdata
from Python_Files import data
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [2]:
upstream_id = "10155200"
downstream_id = "10155500"
start_date = '2016-01-01'
end_date = '2025-12-31'
sites = [upstream_id,downstream_id]

discharge = data.get_discharge(upstream_id,downstream_id,start_date,end_date)
#width = data.get_width()

In [3]:
width = data.get_width(upstream_id,downstream_id,start_date,end_date)
width.head()

,Downstream Width (ft),Upstream Width (ft)
Date,,
2016-01-12,NaN,65.0
2016-01-14,57.0,NaN
2016-02-26,58.0,65.0
2016-04-06,58.5,NaN
2016-04-07,NaN,65.0


In [4]:
g2d = data.get_depth_2_datum(upstream_id,downstream_id,start_date,end_date)

In [7]:
# possible_validation = pd.merge(discharge,cleaning,how='inner',on='Date').dropna()
# sample_daily["Downstream Width (ft)"] = sample_daily["Downstream Width (ft)"].ffill().bfill()
# possible_validation.head(30)

In [8]:
datum, up_MSE, down_MSE = data.log_transform(discharge,'depth')
print(f"Upstream Gauge Depth Training MSE: {up_MSE:.4f}")
print(f"Downstream Gauge Depth Training MSE: {down_MSE:.4f}")

w, up_MSE, down_MSE = data.log_transform(discharge,"width")
print(f"Upstream Width Training MSE: {up_MSE:.4f}")
print(f"Downstream Width Training MSE: {down_MSE:.4f}")

w = w.drop(columns=["Downstream Mean Discharge (cfs)","Upstream Mean Discharge (cfs)"],axis=1)


Upstream Gauge Depth Training MSE: 0.0018
Downstream Gauge Depth Training MSE: 0.0007


TypeError: get_width() missing 4 required positional arguments: 'up_ID', 'down_ID', 'start', and 'end'

In [5]:
start = datetime.strptime(start_date, "%Y-%m-%d")
end = datetime.strptime(end_date, "%Y-%m-%d")
years = np.array(range(start.year, end.year + 1)).reshape(-1, 2)
surface = data.get_surface(upstream_id,downstream_id,years)

In [8]:
#merge and add the to surface and to gauge values
data = pd.merge(discharge,width,how="inner",left_index=True, right_index=True)
depth = pd.merge(surface,g2d,how='inner',on="Date")

data['Upstream Depth (ft)'] = depth['Upstream to Datum (ft)'] + depth['Upstream Gauge Depth (ft)']
data['Downstream Depth (ft)'] = depth['Downstream to Datum (ft)'] + depth['Downstream Gauge Depth (ft)']
data.head()
data.to_csv('../Data Files/Raw/Total_Data.csv')

In [10]:
#only data sets with true values, no filling
validation_set = data.dropna()
validation_set.to_csv('../Data Files/Raw/Validation.csv')